# Data Loading and Preparation

## Basic Imports

In [ ]:
# Data handling
import pandas as pd

# Pathing
from pathlib import Path

# Persistence
import joblib

## Load Flagged Flow Dataset

In [ ]:
flagged_flows_path = Path("../data/processed/stage1_flagged_flows.csv")
flagged_flows = pd.read_csv(flagged_flows_path, low_memory=False)
ff_features = flagged_flows.copy() # Working copy for feature engineering
ff_features.head()

# Time-Based Features

## Start Hour of Day

An `Shourofday` feature is created by extracting the `hour` attribute from the `Stime` column.
- Highlights attacks that occur outside normal working hours.

In [ ]:
Sdt = pd.to_datetime(flagged_flows["Stime"], unit="s")
ff_features["Shourofday"] = Sdt.dt.hour

## Last Hour of Day

An `Lhourofday` feature is created by extracting the `hour` attribute from the `Ltime` column.
- Highlights attacks that occur outside normal working hours.

In [ ]:
Ldt = pd.to_datetime(flagged_flows["Ltime"], unit="s")
ff_features["Lhourofday"] = Ldt.dt.hour

# TTL Features

## TTL Difference Bin

A binary `ttl_diff_bin` feature is created from `ttl_diff` values.
- 1 indicates values above 250 (TP-like).
- 0 indicates values 250 or below (FP-like).
- Provides a clear signal for the model to distinguish true positives from false positives.

In [ ]:
ff_features["ttl_diff_bin"] = (ff_features["ttl_diff"] > 250).astype(int)

## TTL Difference Low

A binary `ttl_diff_low` feature is created from `ttl_diff` values.
- 1 indicates low values between 200-250 (likely true positives).
- 0 indicates values outside that range (FP-like).
- Provides a second categorical cue for TPs just below the main cluster.

In [ ]:
# tp_low_threshold = 200
# ff_features["ttl_diff_low"] = ((ff_features["ttl_diff"] > tp_low_threshold) & (ff_features["ttl_diff"] <= 250)).astype(int)

This feature was ultimately redundant: the numeric `ttl_diff` already captured the necessary signal, and XGBoost did not leverage the binary cue.

## STTL X TTL Difference

A `sttl_x_ttl_diff` feature is created by taking the product of the `sttl` and `ttl_diff` columns.
- Highlights unusual combinations that may be indicative of anomalies.

In [ ]:
ff_features["sttl_x_ttl_diff"] = ff_features["sttl"]*ff_features["ttl_diff"]

## STTL X CT_State_TTL

A `sttl_x_ct_state_ttl` feature is created by taking the product of the `sttl` and `ct_state_ttl` columns.
- Captures how unusual a packet's TTL is given its state frequency.
- Signals rare of suspicious TTL-state combinations.

In [ ]:
ff_features["sttl_x_ct_state_ttl"] = ff_features["sttl"]*ff_features["ct_state_ttl"]

# Categorical Features

## Replace Infrequent Categories With "Other"

In [ ]:
top_k = 10 # Keep 10 most frequent categories
for col in ["state", "service", "dsport"]:
    counts = ff_features[col].value_counts()
    top_categories = counts.nlargest(top_k).index
    ff_features[col] = ff_features[col].where(ff_features[col].isin(top_categories), other="other")

# Export Final Feature Set

## Select Features

### Define List

In [ ]:
features = [
    "dur",
    "Sintpkt",
    "sttl",
    "ct_state_ttl",
    "dbytes",
    "ttl_diff",
    "ttl_diff_bin",
    "sbytes",
    "Djit",
    "anomaly_score",
    "Dload",
    "Lhourofday",
    "state",
    "service",
    "dsport",
    "proto",
    "sttl_x_ct_state_ttl",
    "Shourofday",
    "sttl_x_ttl_diff",
    "smeansz",
    "is_sm_ips_ports"
]

### Save Features to Disk

In [ ]:
features_path = Path("../data/processed/features.pkl")
joblib.dump(features, features_path)
print(f"Features saved to {features_path}.")

## Save Subset to CSV

In [ ]:
ff_features_path = Path("../data/processed/ff_features.csv")
ff_features[features + ["Label"]].to_csv(ff_features_path, index=False)